# Gold Layer – Star Schema & KPIs

This notebook builds the **gold layer** from silver data:
1. **Star schema**: `fact_order_items`, `dim_customer`, `dim_product`, `dim_seller`, `dim_date`
2. **KPIs** computed from the gold tables:
   - KPI 1: Monthly revenue & month-over-month growth
   - KPI 2: Top 3 product categories per state
   - KPI 3: Average delivery delay per seller
   - KPI 4: Repeat-customer rate
   - KPI 5: Cumulative revenue over time
   - KPI 6: Revenue by payment type

## 0 — Setup

In [ ]:
from spark_windows import configure_windows_hadoop

configure_windows_hadoop()

In [ ]:
from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F

spark = (
    SparkSession.builder.master("local[4]")
    .appName("Gold Layer")
    .config("spark.sql.shuffle.partitions", "8")
    .getOrCreate()
)

In [ ]:
SILVER = "./data/silver"
GOLD = "./data/gold"

## 1 — Read Silver Tables

In [ ]:
df_orders = spark.read.parquet(f"{SILVER}/olist_orders_dataset.parquet")
df_customers = spark.read.parquet(f"{SILVER}/olist_customers_dataset.parquet")
df_payments = spark.read.parquet(f"{SILVER}/olist_order_payments_dataset.parquet")
df_products = spark.read.parquet(f"{SILVER}/olist_products_dataset.parquet")
df_sellers = spark.read.parquet(f"{SILVER}/olist_sellers_dataset.parquet")
df_order_items = spark.read.parquet(f"{SILVER}/olist_order_items_dataset.parquet")
df_reviews = spark.read.parquet(f"{SILVER}/olist_order_reviews_dataset.parquet")

print("Silver tables loaded:")
for name, df in [
    ("orders", df_orders), ("customers", df_customers),
    ("payments", df_payments), ("products", df_products),
    ("sellers", df_sellers), ("order_items", df_order_items),
    ("reviews", df_reviews),
]:
    print(f"  {name:15s}  {df.count():>8,} rows")

## 2 — Dimension Tables

### dim_customer

In [ ]:
dim_customer = (
    df_customers
    .select(
        F.monotonically_increasing_id().alias("customer_sk"),
        "customer_id",
        "customer_unique_id",
        "customer_zip_code_prefix",
        "customer_city",
        "customer_state",
    )
)
dim_customer.show(5)
dim_customer.write.mode("overwrite").parquet(f"{GOLD}/dim_customer.parquet")
print(f"✓ dim_customer: {dim_customer.count():,} rows")

### dim_product

In [ ]:
dim_product = (
    df_products
    .select(
        F.monotonically_increasing_id().alias("product_sk"),
        "product_id",
        "product_category_name",
        "product_weight_g",
        "product_length_cm",
        "product_height_cm",
        "product_width_cm",
    )
)
dim_product.show(5)
dim_product.write.mode("overwrite").parquet(f"{GOLD}/dim_product.parquet")
print(f"✓ dim_product: {dim_product.count():,} rows")

### dim_seller

In [ ]:
dim_seller = (
    df_sellers
    .select(
        F.monotonically_increasing_id().alias("seller_sk"),
        "seller_id",
        "seller_zip_code_prefix",
        "seller_city",
        "seller_state",
    )
)
dim_seller.show(5)
dim_seller.write.mode("overwrite").parquet(f"{GOLD}/dim_seller.parquet")
print(f"✓ dim_seller: {dim_seller.count():,} rows")

### dim_date
One row per distinct date in the orders dataset.

In [ ]:
dates = (
    df_orders
    .select(F.col("order_purchase_timestamp").cast("date").alias("date"))
    .filter(F.col("date").isNotNull())
    .distinct()
)

dim_date = (
    dates
    .select(
        F.col("date").alias("date_key"),
        F.year("date").alias("year"),
        F.month("date").alias("month"),
        F.quarter("date").alias("quarter"),
        F.dayofweek("date").alias("day_of_week"),
        F.dayofmonth("date").alias("day_of_month"),
        F.date_format("date", "yyyy-MM").alias("year_month"),
        F.when(F.dayofweek("date").isin(1, 7), True).otherwise(False).alias("is_weekend"),
    )
    .orderBy("date_key")
)
dim_date.show(10)
dim_date.write.mode("overwrite").parquet(f"{GOLD}/dim_date.parquet")
print(f"✓ dim_date: {dim_date.count():,} rows")

## 3 — Fact Table: `fact_order_items`

Grain: one row per item sold.
Includes price, freight, delivery delay, and foreign keys to all dimensions.

In [ ]:
fact_order_items = (
    df_order_items
    .join(df_orders.select(
        "order_id", "customer_id", "order_status",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    ), on="order_id", how="inner")
    .join(dim_customer.select("customer_sk", "customer_id"), on="customer_id", how="left")
    .join(dim_product.select("product_sk", "product_id"), on="product_id", how="left")
    .join(dim_seller.select("seller_sk", "seller_id"), on="seller_id", how="left")
    .withColumn("purchase_date", F.col("order_purchase_timestamp").cast("date"))
    .join(dim_date.select(F.col("date_key").alias("purchase_date"), F.col("date_key").alias("date_sk")),
          on="purchase_date", how="left")
    .withColumn(
        "delivery_delay_days",
        F.datediff(
            F.col("order_delivered_customer_date"),
            F.col("order_estimated_delivery_date"),
        ),
    )
    .select(
        "order_id",
        "order_item_id",
        "customer_sk",
        "product_sk",
        "seller_sk",
        "date_sk",
        "order_status",
        "price",
        "freight_value",
        "delivery_delay_days",
        "order_purchase_timestamp",
        "order_delivered_customer_date",
        "order_estimated_delivery_date",
    )
)

fact_order_items.show(5)
fact_order_items.write.mode("overwrite").parquet(f"{GOLD}/fact_order_items.parquet")
print(f"✓ fact_order_items: {fact_order_items.count():,} rows")

---
## 4 — KPIs

All KPIs below are computed from the gold-layer tables only.

### KPI 1: Monthly Revenue & Month-over-Month Growth

In [ ]:
# Join delivered orders with payments to get revenue per month
orders_delivered = (
    df_orders
    .filter(
        (F.col("order_status") == "delivered")
        & F.col("order_purchase_timestamp").isNotNull()
    )
    .select("order_id", "order_purchase_timestamp")
)

payments_agg = (
    df_payments
    .groupBy("order_id")
    .agg(F.sum("payment_value").alias("payment_value"))
)

orders_with_payments = orders_delivered.join(payments_agg, on="order_id", how="inner")

w_lag = Window.orderBy("month")

kpi_monthly_revenue = (
    orders_with_payments
    .withColumn("month", F.date_format("order_purchase_timestamp", "yyyy-MM"))
    .groupBy("month")
    .agg(F.round(F.sum("payment_value"), 2).alias("revenue"))
    .withColumn("prev_revenue", F.lag("revenue").over(w_lag))
    .withColumn(
        "mom_growth_pct",
        F.round((F.col("revenue") - F.col("prev_revenue")) / F.col("prev_revenue") * 100, 2),
    )
    .orderBy("month")
)

kpi_monthly_revenue.show(30, truncate=False)
kpi_monthly_revenue.write.mode("overwrite").parquet(f"{GOLD}/kpi_monthly_revenue.parquet")
print("✓ KPI 1: Monthly revenue saved")

### KPI 2: Top 3 Product Categories per State

In [ ]:
# Join fact with dimensions to get state + category
items_with_dims = (
    fact_order_items
    .join(dim_customer.select("customer_sk", "customer_state"), on="customer_sk", how="inner")
    .join(dim_product.select("product_sk", "product_category_name"), on="product_sk", how="inner")
)

revenue_by_state_cat = (
    items_with_dims
    .groupBy("customer_state", "product_category_name")
    .agg(F.round(F.sum("price"), 2).alias("total_revenue"))
)

w_rank = Window.partitionBy("customer_state").orderBy(F.desc("total_revenue"))

kpi_top3_categories = (
    revenue_by_state_cat
    .withColumn("rank", F.row_number().over(w_rank))
    .filter(F.col("rank") <= 3)
    .orderBy("customer_state", "rank")
)

kpi_top3_categories.show(30, truncate=False)
kpi_top3_categories.write.mode("overwrite").parquet(f"{GOLD}/kpi_top3_categories_per_state.parquet")
print("✓ KPI 2: Top 3 categories per state saved")

### KPI 3: Average Delivery Delay per Seller

Positive = late delivery, negative = early delivery.

In [ ]:
kpi_delivery_delay = (
    fact_order_items
    .filter(F.col("delivery_delay_days").isNotNull())
    .join(dim_seller.select("seller_sk", "seller_id", "seller_city", "seller_state"),
          on="seller_sk", how="inner")
    .groupBy("seller_id", "seller_city", "seller_state")
    .agg(
        F.round(F.avg("delivery_delay_days"), 2).alias("avg_delay_days"),
        F.count("*").alias("total_orders"),
        F.round(F.avg(F.when(F.col("delivery_delay_days") > 0, F.col("delivery_delay_days"))), 2)
         .alias("avg_late_days"),
        F.round(
            F.sum(F.when(F.col("delivery_delay_days") > 0, 1).otherwise(0)) / F.count("*") * 100, 2
        ).alias("late_pct"),
    )
    .orderBy(F.desc("avg_delay_days"))
)

kpi_delivery_delay.show(20, truncate=False)
kpi_delivery_delay.write.mode("overwrite").parquet(f"{GOLD}/kpi_delivery_delay_per_seller.parquet")
print("✓ KPI 3: Delivery delay per seller saved")

### KPI 4: Repeat-Customer Rate

**Important**: `customer_id` changes per order. We must use `customer_unique_id`
to identify the actual person across orders.

In [ ]:
orders_per_person = (
    df_orders
    .filter(F.col("order_status") == "delivered")
    .join(df_customers.select("customer_id", "customer_unique_id"), on="customer_id", how="inner")
    .groupBy("customer_unique_id")
    .agg(F.count("order_id").alias("order_count"))
)

total_customers = orders_per_person.count()
repeat_customers = orders_per_person.filter(F.col("order_count") > 1).count()
one_time_customers = total_customers - repeat_customers

kpi_repeat_rate = spark.createDataFrame([
    ("total_unique_customers", total_customers),
    ("one_time_customers", one_time_customers),
    ("repeat_customers", repeat_customers),
    ("repeat_rate_pct", round(repeat_customers / total_customers * 100, 2) if total_customers > 0 else 0.0),
], ["metric", "value"])

kpi_repeat_rate.show(truncate=False)

# Also build a distribution of order counts
order_count_dist = (
    orders_per_person
    .groupBy("order_count")
    .agg(F.count("*").alias("num_customers"))
    .orderBy("order_count")
)
order_count_dist.show(20)

kpi_repeat_rate.write.mode("overwrite").parquet(f"{GOLD}/kpi_repeat_customer_rate.parquet")
order_count_dist.write.mode("overwrite").parquet(f"{GOLD}/kpi_order_count_distribution.parquet")
print("✓ KPI 4: Repeat-customer rate saved")

### KPI 5: Cumulative Revenue Over Time

In [ ]:
w_cumulative = Window.orderBy("month").rowsBetween(Window.unboundedPreceding, Window.currentRow)

kpi_cumulative_revenue = (
    kpi_monthly_revenue
    .select("month", "revenue")
    .withColumn("cumulative_revenue", F.round(F.sum("revenue").over(w_cumulative), 2))
)

kpi_cumulative_revenue.show(30, truncate=False)
kpi_cumulative_revenue.write.mode("overwrite").parquet(f"{GOLD}/kpi_cumulative_revenue.parquet")
print("✓ KPI 5: Cumulative revenue saved")

### KPI 6: Revenue by Payment Type

In [ ]:
kpi_payment_type = (
    df_payments
    .join(
        df_orders
        .filter(F.col("order_status") == "delivered")
        .select("order_id"),
        on="order_id",
        how="inner",
    )
    .groupBy("payment_type")
    .agg(
        F.round(F.sum("payment_value"), 2).alias("total_revenue"),
        F.count("*").alias("transaction_count"),
        F.round(F.avg("payment_value"), 2).alias("avg_transaction_value"),
        F.round(F.avg("payment_installments"), 1).alias("avg_installments"),
    )
    .withColumn(
        "revenue_share_pct",
        F.round(
            F.col("total_revenue") / F.sum("total_revenue").over(Window.partitionBy()) * 100, 2
        ),
    )
    .orderBy(F.desc("total_revenue"))
)

kpi_payment_type.show(truncate=False)
kpi_payment_type.write.mode("overwrite").parquet(f"{GOLD}/kpi_revenue_by_payment_type.parquet")
print("✓ KPI 6: Revenue by payment type saved")

## 5 — Summary

| # | KPI | Gold Table |
|---|-----|-----------|
| 1 | Monthly revenue & MoM growth | `kpi_monthly_revenue` |
| 2 | Top 3 categories per state | `kpi_top3_categories_per_state` |
| 3 | Avg delivery delay per seller | `kpi_delivery_delay_per_seller` |
| 4 | Repeat-customer rate | `kpi_repeat_customer_rate` |
| 5 | Cumulative revenue over time | `kpi_cumulative_revenue` |
| 6 | Revenue by payment type | `kpi_revenue_by_payment_type` |

In [ ]:
print("\n" + "=" * 60)
print("  GOLD LAYER COMPLETE")
print("=" * 60)

import os
gold_files = [f for f in os.listdir(GOLD) if f.endswith(".parquet")]
for f in sorted(gold_files):
    print(f"  📦 {f}")

print(f"\n  Total gold tables: {len(gold_files)}")